# P12 — Entrenar modelos de lenguaje para seguir instrucciones con retroalimentación humana

## 1. Título y paper

**Paper:** *Training language models to follow instructions with human feedback*  
**Autoría:** Long Ouyang, Jeff Wu, Xu Jiang, Diogo Almeida, y otros (OpenAI)  
**Año y venue:** 2022 · arXiv:2203.02155 · NeurIPS 2022  
**Nivel:** L3 · **Motor:** `rlhf`  
**Ficha completa:** [`P12_instructgpt_rlhf`](../../papers/foundational/P12_instructgpt_rlhf/README.md)

**Hito:** El salto de «modelo que completa texto» a «asistente que sigue instrucciones»: alineación con preferencias humanas.

- [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Maximizar la verosimilitud del texto de internet no es lo mismo que ser útil, honesto e inocuo; el objetivo de entrenamiento está desalineado con la intención del usuario.
2. Ejecutar una implementación mínima de la propuesta: Tres etapas: ajuste supervisado con demostraciones, modelo de recompensa entrenado con comparaciones humanas y optimización por PPO con penalización KL.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P10
- Christiano et al. (2017), aprendizaje por refuerzo con preferencias humanas


## 4. Intuición

Es mucho más fácil decir «prefiero esta respuesta a esta otra» que puntuar del 1 al 10. RLHF convierte miles de esas comparaciones en un número que el modelo puede optimizar.


## 5. Concepto mínimo

Tres etapas:

```text
1. SFT : ajuste supervisado con demostraciones humanas
2. RM  : modelo de recompensa Bradley-Terry, p(y_w ≻ y_l) = σ(r(y_w) − r(y_l))
3. RL  : maximizar r(y) − β·KL(π ‖ π_SFT) con PPO
```

El término KL impide que la política se aleje tanto del modelo base que empiece a producir texto degenerado con recompensa alta.


## 6. Código explicado

El motor ajusta el modelo de recompensa sobre 5 comparaciones y reordena las respuestas candidatas.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('rlhf', seed=7)['result']
show(r['pesos_del_modelo_de_recompensa'])
print('\nranking aprendido:')
for fila in r['ranking_aprendido']:
    print(f"  r={fila['reward']:+7.3f} · {fila['texto']}")

## 7. Predicción antes de ejecutar

1. ¿Qué característica recibirá más peso: utilidad, honestidad, inocuidad o verbosidad?
2. ¿Quedará la respuesta peligrosa (d) arriba o abajo del ranking?
3. Si todas las respuestas preferidas fueran también las más largas, ¿qué aprendería el modelo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
r7 = run_paper_lab('rlhf', seed=7)['result']
pesos = r7['pesos_del_modelo_de_recompensa']
orden = sorted(pesos.items(), key=lambda kv: -abs(kv[1]))
print('características por influencia absoluta:')
for nombre, peso in orden:
    print(f'  {nombre:<12} {peso:+.3f}')

## 9. Salida interpretable

«inocuidad» y «utilidad» dominan porque las comparaciones que le dimos castigan lo peligroso y premian lo útil. **El modelo de recompensa no descubre valores: reproduce los del conjunto de comparaciones.** Cambia las comparaciones y cambian los valores.


## 10. Comentario pedagógico

Aquí está el punto político y técnico a la vez: quién etiqueta, con qué guía y con qué incentivos determina qué significa «mejor». El propio paper documenta el perfil de sus anotadores; leer esa sección es parte del ejercicio.


## 11. Error o anti-patrón deliberado

Anti-patrón: reward hacking. Si «verbosidad» correlaciona con preferencia, la política aprende a ser larga, no mejor.


In [ ]:
preferencias_sesgadas = [('largo', 'corto')] * 5
print('Si TODAS las preferencias premian la respuesta larga:')
print('  el modelo de recompensa aprende r ∝ longitud')
print('  la política optimiza longitud')
print('  la métrica sube y la calidad real no se mueve')

## 12. Corrección

Mitigaciones: penalización KL contra el modelo base, comparaciones controladas por longitud y evaluación humana independiente del RM.


In [ ]:
import math

def objetivo(recompensa, kl, beta=0.2):
    return recompensa - beta * kl

for kl in (0.0, 1.0, 5.0, 20.0):
    print(f'KL={kl:>5.1f} → objetivo con r=3.0: {objetivo(3.0, kl):+.2f}')
print('→ alejarse del modelo base se vuelve caro: eso frena el reward hacking extremo')

## 13. Desafío guiado

Invierte una comparación (haz que se prefiera la respuesta evasiva) y observa cómo se reordena todo.


In [ ]:
print('pesos originales:', run_paper_lab('rlhf', seed=1)['result']['pesos_del_modelo_de_recompensa'])
print('pesos con otra semilla:', run_paper_lab('rlhf', seed=99)['result']['pesos_del_modelo_de_recompensa'])
print('→ los pesos son estables porque las COMPARACIONES son las mismas;')
print('  la semilla no cambia los datos de preferencia, y eso es lo que manda.')

## 14. Desafío autónomo

Crea 30 pares de preferencia propios sobre un dominio que conozcas. Entrena el modelo de recompensa, y luego construye adversarialmente una respuesta que maximice la recompensa siendo claramente peor. Documenta qué característica explotaste.


## 15. Evidencia de aprendizaje

Guarda los pesos aprendidos, el ranking, tu ejemplo de reward hacking y la explicación del papel del término KL.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P12_instructgpt_rlhf/README.md) · evaluación formal: [`assessments/papers/P12_instructgpt_rlhf.md`](../../assessments/papers/P12_instructgpt_rlhf.md)


## 16. Cierre

El asistente ya sigue instrucciones. La siguiente pregunta es si hace falta todo el aparato de RL para conseguirlo — la respuesta llega en P15.


## 17. Conexión con el siguiente hito

- P15

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
